# Bluff Detector: run M1, RAG

Runs **M1, RAG** (the base model, with the 3 most relevant passages from the Kenyan document collection pasted into every prompt) on a free Colab T4 GPU and saves everything to your private Hugging
Face dataset repo, so nothing is lost when the Colab session ends. Tests: **H4 (vs M0)**.
Time: ~6 hours on a T4 (the passages make every prompt about 2,000 tokens).

**Nothing to edit.** Top right: **Select Kernel → Colab → New Colab Server → GPU → T4**, then
**Run All**. Paste the GitHub token, then the Hugging Face token, when asked.

**If the session drops:** Ctrl+Shift+P → Reload Window → reconnect → **Run All** again. It picks
up where it stopped and only asks the questions not yet answered.

**On Kaggle instead** (when Colab refuses a GPU): kaggle.com → **Create → New Notebook → File →
Import Notebook** (this file) → right panel: **Accelerator = GPU T4 x2**, **Internet = On** →
**Add-ons → Secrets**: `GH_TOKEN` and `HF_TOKEN`, both attached → **Save Version → Save & Run
All (Commit)**. It runs in the background; the laptop can be closed.

*Made by `scripts/make_run_notebooks.py` from `colab_run.ipynb`: don't edit this file by hand.*

In [ ]:
CONFIG = "configs/variants/m1_rag.yaml"  # fixed: this notebook only runs m1_rag
HF_REPO = "vuhyeahhh/bluff-detector-runs"  # your private Hugging Face dataset repo

## 1. Check the GPU

In [ ]:
import os

# One GPU only: Kaggle's "T4 x2" would otherwise split the model across two cards. Set before
# torch starts, and inherited by every `bluff` command below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import torch  # noqa: E402

assert torch.cuda.is_available(), "No GPU: Colab: New Colab Server -> T4; Kaggle: GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Tokens (Kaggle Secrets, or typed in; never saved in this notebook)

In [ ]:
ON_KAGGLE = os.path.isdir("/kaggle/working")
if ON_KAGGLE:  # Add-ons -> Secrets: GH_TOKEN and HF_TOKEN, attached to this notebook
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    GH_TOKEN, HF_TOKEN = secrets.get_secret("GH_TOKEN"), secrets.get_secret("HF_TOKEN")
else:
    from getpass import getpass

    GH_TOKEN = getpass("GitHub token (read-only, this repo): ")
    HF_TOKEN = getpass("Hugging Face token (write): ")
assert GH_TOKEN and HF_TOKEN, "a token is empty: paste it again"

## 3. Get the code and install it

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/bluff-detector" if ON_KAGGLE else "/content/bluff-detector"
url = f"https://{GH_TOKEN}@github.com/trevorblue/bluff-detector.git"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", url, "main"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run("pip install -q -e '.[llm,data]' huggingface_hub", shell=True, check=True)
# A package installed while Python runs is seen only after a restart; this makes it visible now.
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 4. Bring back saved progress (if this version ran before)

In [ ]:
from huggingface_hub import HfApi, snapshot_download

from bluff.config import load_config

VARIANT = load_config(CONFIG)["variant"]["id"]
api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
snapshot_download(HF_REPO, repo_type="dataset", token=HF_TOKEN, local_dir=".",
                  allow_patterns=[f"data/interim/{VARIANT}/*", "artefacts/registry/**"])
done = os.path.exists(f"data/interim/{VARIANT}/generations.jsonl")
print(VARIANT, "- resuming saved progress" if done else "- starting fresh")

def save(folders):
    for folder in folders:
        if os.path.isdir(folder):
            api.upload_folder(repo_id=HF_REPO, repo_type="dataset", folder_path=folder,
                              path_in_repo=folder, commit_message=f"{VARIANT}: {folder}")

## 5. Ask the questions (the GPU part), saving every 10 minutes

In [ ]:
import time


def step(stage):
    r = subprocess.run(["bluff", stage, "--config", CONFIG], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-4000:])
        raise RuntimeError(f"{stage} failed: the reason is printed above")
    print(stage, "done")

for stage in ("validate", "ingest", "audit"):
    step(stage)
log = open("generate.log", "w")
proc = subprocess.Popen(["bluff", "generate", "--config", CONFIG], stdout=log, stderr=log)
last = time.time()
while proc.poll() is None:
    time.sleep(30)
    if time.time() - last > 600:
        save([f"data/interim/{VARIANT}"])
        print(time.strftime("%H:%M"), "saved |", open("generate.log").read().splitlines()[-1:])
        last = time.time()
log.close()
print(open("generate.log").read()[-2000:])
assert proc.returncode == 0, "generate failed: see the log above"
save([f"data/interim/{VARIANT}"])


## 6. Label, train, evaluate, register, export (CPU, a few minutes)

In [ ]:
for stage in ("label", "featurize", "train", "evaluate", "register", "export"):
    step(stage)
save([f"data/interim/{VARIANT}", "data/processed", f"artefacts/{VARIANT}",
      "artefacts/registry", f"reports/{VARIANT}", f"data/datasets/{VARIANT}"])
print(open(f"reports/{VARIANT}/summary.md").read())

## 6b. Finish quickly on Kaggle

After the last cell, Kaggle copies everything in `/kaggle/working` into the run's Output, which
took over an hour after M2 while the GPU clock kept running. Everything is on Hugging Face by
now (the cell above raises an error if an upload fails), so the working copy is removed first.

In [ ]:
if ON_KAGGLE:
    import shutil

    os.chdir("/kaggle/working")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    print("Working copy removed: the session can end as soon as Kaggle notices the run is done.")
else:
    print("Not on Kaggle: nothing to clean up.")

## 7. On your laptop

Everything is now in your Hugging Face repo. Download it into the project folder with:

```
cd bluff-detector
hf download vuhyeahhh/bluff-detector-runs --repo-type dataset --local-dir . --exclude .gitattributes
```